# Romanisation Pipeline (IndicXlit)

> **Paper:** *Lost in Transliteration: Orthographic Sensitivity in Neural MT Evaluation*  
> **Authors:** G L John Salvin · Swapnil Hingmire · IIT Palakkad · 2026  
> **Notebook:** `02_romanisation_pipeline.ipynb`

---

This notebook applies **IndicXlit** (Madhani et al., 2023) to the target side of the IndicMT Eval dataset, converting native-script translations and references into Latin script.

The intervention is **script-only**: all phonological, morphological, and semantic content is preserved. Source sentences and MQM scores are left untouched, so any downstream change in a metric score is attributable exclusively to orthographic form.

Supported language codes:

| Language | ISO code | Native script | IndicXlit code |
|---|---|---|---|
| Hindi | HIN | Devanagari | `hi` |
| Marathi | MAR | Devanagari | `mr` |
| Gujarati | GUJ | Gujarati | `gu` |
| Tamil | TAM | Tamil | `ta` |
| Malayalam | MAL | Malayalam | `ml` |

**Output columns added to each CSV:**
- `Reference_Romanised` — romanised reference sentence
- `Translation_Romanised` — romanised MT hypothesis


## Setup

Install dependencies if needed:

```bash
pip install ai4bharat-transliteration pandas openpyxl
```

IndicXlit is loaded once and reused across all five languages. Each language requires its own `XlitEngine` instance initialised with the corresponding ISO 639-1 code.

**Data path:** place the five per-language CSV files in `../data/processed/` before running. Romanised outputs will be written to `../data/processed/romanised_outputs/` and also exported as a combined Excel workbook.


In [ ]:
import os
import glob
import pandas as pd
from ai4bharat.transliteration import XlitEngine

DATASET_DIR = "../data/processed"
OUTPUT_DIR  = "../data/processed/romanised_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# columns on the target side that should be romanised
TARGET_COLUMNS = ["Reference", "Translation"]

# mapping: keyword in filename -> (language display name, IndicXlit ISO code)
LANGUAGE_MAP = {
    "hindi":     ("Hindi",     "hi"),
    "marathi":   ("Marathi",   "mr"),
    "gujarati":  ("Gujarati",  "gu"),
    "tamil":     ("Tamil",     "ta"),
    "malayalam": ("Malayalam", "ml"),
}

csv_files = sorted(glob.glob(os.path.join(DATASET_DIR, "*.csv")))
print(f"Found {len(csv_files)} CSV file(s):")
for f in csv_files:
    print(f"  {os.path.basename(f)}")


## Romanising Each Language File

For each CSV file we:

1. Detect the language from the filename.
2. Initialise an `XlitEngine` for that language (loaded once per file).
3. Transliterate every non-null cell in `Reference` and `Translation`.
4. Insert the romanised text as a new column immediately after the source column.

IndicXlit returns the single best transliteration by default (`topk=1`). Empty or NaN cells are passed through as empty strings.

The `resolve_columns` helper from `01_tokenization_parity.ipynb` is reproduced here for case-insensitive, punctuation-agnostic column matching, since column names vary slightly across the five language files.


In [ ]:
def detect_language(filename):
    for keyword, (lang_name, lang_code) in LANGUAGE_MAP.items():
        if keyword in filename.lower():
            return lang_name, lang_code
    return "Unknown", None


def resolve_columns(df, targets):
    """Match target names to actual DataFrame columns, ignoring case and punctuation."""
    def norm(s):
        return "".join(ch.lower() for ch in s if ch.isalnum())
    col_map = {norm(c): c for c in df.columns}
    return {t: col_map[norm(t)] for t in targets if norm(t) in col_map}


def transliterate_series(series, engine, lang_code):
    """Transliterate a pandas Series using IndicXlit."""
    results = []
    for text in series:
        if text is None or (isinstance(text, float) and pd.isna(text)) or str(text).strip() == "":
            results.append("")
            continue
        try:
            output = engine.translit_sentence(str(text), lang_code)
            results.append(output)
        except Exception:
            results.append("")
    return pd.Series(results, index=series.index)


summary_rows = []

for csv_path in csv_files:
    base_name  = os.path.splitext(os.path.basename(csv_path))[0]
    lang_name, lang_code = detect_language(base_name)

    if lang_code is None:
        print(f"Could not detect language for: {base_name} — skipped\n")
        continue

    print(f"{'='*65}")
    print(f"Language : {lang_name} ({lang_code})")
    print(f"File     : {base_name}")
    print(f"{'='*65}")

    df = pd.read_csv(csv_path)

    print(f"Loading IndicXlit engine for: {lang_code}")
    engine = XlitEngine(lang_code, beam_width=4, rescore=True)
    print("Engine ready.\n")

    resolved = resolve_columns(df, TARGET_COLUMNS)
    missing  = set(TARGET_COLUMNS) - set(resolved.keys())
    if missing:
        print("  Columns not found in this file (skipped):")
        for m in sorted(missing):
            print(f"    - {m}")
        print()

    cols_to_insert = []

    for requested, actual in resolved.items():
        out_col = f"{actual}_Romanised"
        if out_col in df.columns:
            print(f"  '{out_col}' already exists — skipped")
            continue

        print(f"  Romanising column: '{actual}' -> '{out_col}'")
        romanised = transliterate_series(df[actual], engine, lang_code)

        non_empty  = romanised.str.strip().str.len().gt(0).sum()
        print(f"  checkmark {non_empty}/{len(df)} rows romanised successfully")

        insert_at = df.columns.get_loc(actual) + 1
        cols_to_insert.append((insert_at, out_col, romanised))

        summary_rows.append({
            "Language":   lang_name,
            "Column":     actual,
            "Output Col": out_col,
            "Rows":        len(df),
            "Romanised":  non_empty,
            "Empty":       len(df) - non_empty,
        })

    for insert_at, col_name, series in sorted(cols_to_insert, key=lambda x: x[0], reverse=True):
        df.insert(insert_at, col_name, series)

    out_path = os.path.join(OUTPUT_DIR, f"{base_name}_romanised.csv")
    df.to_csv(out_path, index=False)
    print(f"\n  -> Saved: {out_path}\n")

print(f"{'='*65}")
print("SUMMARY — Romanisation Coverage")
print(f"{'='*65}")
if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_string(index=False))
    summary_path = os.path.join(OUTPUT_DIR, "romanisation_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"\nSummary saved: {summary_path}")
print("\nAll done.")


## Sanity Check — Side-by-Side Samples

Print five random rows per language showing the native and romanised forms of the MT hypothesis. This is a manual check to confirm the transliteration is phonologically consistent; no automated assertion is made here since IndicXlit output is deterministic for a given model checkpoint.


In [ ]:
rom_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*_romanised.csv")))

N_SAMPLES = 5

for path in rom_files:
    df   = pd.read_csv(path)
    lang = detect_language(os.path.basename(path))[0]

    native_col = next((c for c in df.columns
                       if c.lower() == "translation"), None)
    roman_col  = next((c for c in df.columns
                       if "translation" in c.lower() and "romanised" in c.lower()), None)

    if not native_col or not roman_col:
        print(f"Could not find Translation columns in {os.path.basename(path)} — skipped\n")
        continue

    sample = df[[native_col, roman_col]].dropna().sample(
        min(N_SAMPLES, len(df)), random_state=42
    )

    print(f"{'='*65}")
    print(f"{lang} — Translation (native) vs Translation_Romanised")
    print(f"{'='*65}")
    for _, row in sample.iterrows():
        print(f"  Native   : {row[native_col]}")
        print(f"  Romanised: {row[roman_col]}")
        print()


## Exporting to Excel

All five romanised CSV files are combined into a single Excel workbook — one sheet per language. This file, together with the native-script CSV files, feeds into `03_reproduce_wmt24_core.ipynb` and the subsequent metric-calculation notebooks.


In [ ]:
excel_path = os.path.join(OUTPUT_DIR, "romanised_outputs_all.xlsx")
rom_csvs   = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*_romanised.csv")))

if not rom_csvs:
    print(f"No romanised CSV files found in: {OUTPUT_DIR}")
else:
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        for csv_path in rom_csvs:
            sheet_name = os.path.splitext(os.path.basename(csv_path))[0][:31]
            pd.read_csv(csv_path).to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"  checkmark Sheet added: {sheet_name}")
    print(f"\ncheckmark Combined Excel saved to: {excel_path}")


---

## References

**IndicXlit transliteration system:**  
Madhani, Y., Parthan, S., Bedekar, P., Nc, G., Khapra, R., Kunchukuttan, A., Kumar, P., & Khapra, M. (2023). *Aksharantar: Open Indic-Language Transliteration Datasets and Models for the Next Billion Users.* EMNLP 2023 Findings, pp. 40–57. https://aclanthology.org/2023.findings-emnlp.3

**IndicMT Eval dataset:**  
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., & Dabre, R. (2023). *IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics for Indian Languages.* ACL 2023, pp. 14210–14228. https://aclanthology.org/2023.acl-long.795

**Romanagari / Tanglish / Manglish usage:**  
Chavan, T., Sehgal, A., Patil, O., Pujar, S., Galav, R., & Joshi, R. (2023). *My Boli: Code-mixed Marathi-English Corpora, Pretrained Language Models and Evaluation Benchmarks.* IJCNLP-AACL 2023. https://aclanthology.org/2023.ijcnlp-main.67
